# The asymmetric swap gate `PAsymSwap`

!!! note "In brief"

    `PAsymSwap` is the asymmetric stochastic swap gate of arXiv 2608.01615 (Section IV.1, Eq. 33): on one edge the occupied value hops forward with probability $p$ and backward with probability $q$, and $|00)$, $|11)$ are fixed. `PJUMP` is its $q = 0$ case and `PSWAP` its $p = q$ case. This notebook re-runs tutorial 02's directed 5-cycle with `PAsymSwap` in place of `PJUMP`, then adds a backward rate on every edge, and checks in each case that the number of occupied pbits is exactly conserved in every sample.

The companion script `examples/scripts/pasymswap_compile_forms.py` (needs `thrml`) compiles the same gate into Ising kernels two ways and compares what each conserves.

In [ ]:
import jax, jax.numpy as jnp, numpy as np
from jax.scipy.special import logit
from torx.psc import BranchingSimulator, DiscretePCircuit, PAsymSwap, PJUMP

# --- tutorial 02, "Directed transport: the PJUMP gate", verbatim constants ---
FAST, SLOW = 0.9, 0.4
n_nodes = 5
directed_edges = [(0, 1, FAST), (1, 2, FAST), (2, 3, SLOW), (3, 4, FAST), (4, 0, SLOW)]
T, N = 6.0, 300
dt = T / N
snapshot_reps = np.arange(60, N + 1, 60, dtype=int)
initial_bits = jnp.zeros(n_nodes, dtype=jnp.int32).at[0].set(1)
p0 = np.zeros(n_nodes); p0[0] = 1.0

def exact_occ(Q, p0, times):
    vals, vecs = np.linalg.eig(Q); coeff = np.linalg.solve(vecs, p0)
    return np.stack([(vecs @ (np.exp(vals * t) * coeff)).real for t in times])

def run(gates, thetas, num_samples=20_000):
    sim = BranchingSimulator(num_samples=num_samples)
    keys = jax.random.split(jax.random.key(0), len(snapshot_reps))
    occ, mass_min, mass_max = [], 1e9, -1e9
    for reps, key in zip(snapshot_reps, keys, strict=True):
        compiled = sim.build_circuit(DiscretePCircuit(gates, reps=int(reps)), thetas)
        samples = np.asarray(sim.sample(compiled, initial_bits, key))
        occ.append(samples.mean(0)); m = samples.sum(1); mass_min, mass_max = min(mass_min, m.min()), max(mass_max, m.max())
    return np.stack(occ), (mass_min, mass_max)

## Tutorial 02's directed transport, with `PJUMP`

The constants are those of tutorial 02's "Directed transport" cell. The reference is the single-particle CTMC $e^{Qt} p_0$.

In [ ]:
# (1) tutorial circuit, PJUMP, one rate per directed edge (backward rate 0)
Q = np.zeros((n_nodes, n_nodes))
for s, d, r in directed_edges: Q[d, s] += r; Q[s, s] -= r
gates = [PJUMP([int(s), int(d)]) for s, d, r in directed_edges]
thetas = [jnp.array([logit(1.0 - np.exp(-r * dt))]) for s, d, r in directed_edges]
occ_pjump, mass_pjump = run(gates, thetas)
ex = exact_occ(Q, p0, snapshot_reps * dt)
print(f"PJUMP    : max |Torx - exact| = {np.abs(occ_pjump - ex).max():.4f}; particles per sample in [{mass_pjump[0]}, {mass_pjump[1]}]")

## The same circuit with `PAsymSwap`

`PJUMP` is `PAsymSwap` with $q = 0$, i.e. $\theta_1 = -\infty$; the matrices are identical, only the random stream differs.

In [ ]:
# (2) same circuit with PAsymSwap: PJUMP is PAsymSwap with q = 0 (theta_1 = -inf); identical matrices
gates_a = [PAsymSwap([int(s), int(d)]) for s, d, r in directed_edges]
thetas_a = [jnp.array([logit(1.0 - np.exp(-r * dt)), -jnp.inf]) for s, d, r in directed_edges]
for g, ga, th, tha in zip(gates, gates_a, thetas, thetas_a):
    assert jnp.allclose(g.get_matrix(th), ga.get_matrix(tha))
occ_a, mass_a = run(gates_a, thetas_a)
print(f"PAsymSwap(q=0): max |Torx - exact| = {np.abs(occ_a - ex).max():.4f}; particles per sample in [{mass_a[0]}, {mass_a[1]}]")

## A two-directional walk

A backward rate on every edge (Eq. 33 with $p_{ij} \ne p_{ji} \ne 0$). `PAsymSwap.theta_from_probs` converts hop probabilities to logits.

In [ ]:
# (3) a genuinely asymmetric walk: add a backward rate BACK on every edge (Eq. 33 with p_ij != p_ji != 0)
BACK = 0.25
Q2 = Q.copy()
for s, d, r in directed_edges: Q2[s, d] += BACK; Q2[d, d] -= BACK
thetas_b = [PAsymSwap.theta_from_probs(jnp.array(1.0 - np.exp(-r * dt)), jnp.array(1.0 - np.exp(-BACK * dt))) for s, d, r in directed_edges]
occ_b, mass_b = run(gates_a, thetas_b)
ex2 = exact_occ(Q2, p0, snapshot_reps * dt)
print(f"PAsymSwap(p,q): max |Torx - exact| = {np.abs(occ_b - ex2).max():.4f}; particles per sample in [{mass_b[0]}, {mass_b[1]}]")

## Conservation is integer-exact for any parameters

`BranchingSimulator` samples a branch index and applies a permutation within the occupation sector, so the particle count cannot change, whatever $\theta$ is; with several walkers the gate is an exclusion process, as tutorial 02 notes for `PJUMP`.

In [ ]:
# (4) random theta, three walkers: still exactly one particle count for every sample
keys = jax.random.split(jax.random.key(3), len(gates_a))
thetas_r = [3.0 * jax.random.normal(k, (2,)) for k in keys]
x3 = jnp.zeros(n_nodes, dtype=jnp.int32).at[:3].set(1)
sim = BranchingSimulator(num_samples=20_000)
s = np.asarray(sim.sample(sim.build_circuit(DiscretePCircuit(gates_a, reps=300), thetas_r), x3, jax.random.key(9)))
print(f"random theta, 3 walkers, 300 layers: particles per sample in [{s.sum(1).min()}, {s.sum(1).max()}] over {len(s)} samples")

assert s.sum(1).min() == 3 and s.sum(1).max() == 3